# 22 — External sync vLLM GRPO rollout evidence

Этот notebook — рабочий обходной путь для старта GRPO, пока Tunix `RLCluster` ↔ vLLM V1 weight-sync проверяется отдельно. Он использует тот же прямой `VllmInferenceEngine`, что и notebook 17: CrafText task → MegaPrompts → Qwen chat template → sync vLLM batch → batched env step → replay artifacts → `ExternalGrpoBatch` с group-normalized advantages.

Важно: здесь actor weights ещё не обновляются. Эта тетрадка создаёт корректный GRPO evidence batch, который дальше можно передать в внешний updater/LoRA/Qwix или использовать как валидационный слой перед PPO с critic.


In [ ]:
import os

# Must run before importing JAX/vLLM/Tunix. Restart the kernel after changing these.
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'

print('XLA_PYTHON_CLIENT_PREALLOCATE=', os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'])
print('VLLM_WORKER_MULTIPROC_METHOD=', os.environ['VLLM_WORKER_MULTIPROC_METHOD'])


In [ ]:
from pathlib import Path

from tunix_craftext.inference import load_generation_pipeline_config

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError('Run this notebook from inside the tunix-craftext repository')

GENERATION_CONFIG = ROOT / 'configs/inference/vllm/qwen25_05b_sync.yaml'
generation = load_generation_pipeline_config(GENERATION_CONFIG)
SNAPSHOT = Path(generation.profile.model)
if not SNAPSHOT.is_absolute():
    SNAPSHOT = ROOT / SNAPSHOT
if not SNAPSHOT.is_dir():
    raise FileNotFoundError(f'Local model snapshot is missing: {SNAPSHOT}')

CONFIG_PATH = ROOT / 'configs/env/text/qwen_craftext.yaml'
RUN_DIR = ROOT / 'artifacts/runs/external-vllm-grpo'
REPLAY_DIR = RUN_DIR / 'replays'
REPLAY_DIR.mkdir(parents=True, exist_ok=True)
print('repo:', ROOT)
print('generation config:', GENERATION_CONFIG)
print('snapshot:', SNAPSHOT)
print('run dir:', RUN_DIR)


In [ ]:
from tunix_craftext.env.config import load_mvp_config
from tunix_craftext.env.runtime import build_craftext_runtime
from tunix_craftext.env.tasks import CrafTextTaskSampler

config = load_mvp_config(CONFIG_PATH)
runtime = build_craftext_runtime(config)
task_sampler = CrafTextTaskSampler.from_runtime(
    runtime,
    horizon=config.environment.horizon,
    mode='fixed',
    fixed_instruction_index=config.environment.instruction_index,
    goal_prefix=config.run.goal,
)
prepared_goal, prepared_instruction_index = task_sampler.task_at(config.environment.instruction_index)
print('instruction_index:', prepared_instruction_index)
print('goal:', prepared_goal)
print('actions:', runtime.actions.labels)


In [ ]:
from transformers import AutoTokenizer

from tunix_craftext.env.prompts import MegaPromptRenderer, RenderedPrompt

base_renderer = MegaPromptRenderer(config.prompt.template)
tokenizer = AutoTokenizer.from_pretrained(str(SNAPSHOT), trust_remote_code=True)

class QwenChatTemplateRenderer:
    def __init__(self, renderer, tokenizer):
        self.renderer = renderer
        self.tokenizer = tokenizer

    def render(self, context):
        rendered = self.renderer.render(context)
        chat_text = self.tokenizer.apply_chat_template(
            [
                {
                    'role': 'system',
                    'content': 'You are a CrafText agent. Return exactly one <action>LABEL</action>.',
                },
                {'role': 'user', 'content': rendered.text},
            ],
            add_generation_prompt=True,
            tokenize=False,
        )
        if not isinstance(chat_text, str) or not chat_text.strip():
            raise ValueError('Qwen chat template did not return text')
        return RenderedPrompt(chat_text, rendered.actions, rendered.template_name)

renderer = QwenChatTemplateRenderer(base_renderer, tokenizer)


In [ ]:
from dataclasses import replace

from tunix_craftext.inference import RequestsLlmBackend, VllmInferenceEngine

engine_profile = replace(
    generation.profile,
    model=str(SNAPSHOT),
    metadata={**dict(generation.profile.metadata), 'notebook': '22_external_vllm_sync_grpo_rollout.ipynb'},
)
engine = VllmInferenceEngine.from_profile(engine_profile)
backend = RequestsLlmBackend(engine)
print('engine:', engine.profile)
print('tunix rollout kwargs:', generation.tunix.to_tunix_rollout_kwargs())


In [ ]:
import jax

from tunix_craftext.env.prompts import PromptContext
from tunix_craftext.inference import GenerationBatch, collect_generation_results_sync
from tunix_craftext.models.llm import LlmRequest

reset = runtime.adapter.reset(jax.random.PRNGKey(config.run.seed))
smoke_prompt = renderer.render(
    PromptContext(
        prepared_goal,
        runtime.adapter.prompt_state(reset.state),
        runtime.actions,
    )
)
smoke = collect_generation_results_sync(
    engine,
    (
        GenerationBatch(
            (LlmRequest(smoke_prompt, max_new_tokens=min(8, generation.tunix.max_tokens_to_generate)),),
            group_id='external-grpo-smoke',
            policy_version=engine.profile.policy_version,
        ),
    ),
)
print('smoke response:', smoke[0].result.responses[0].raw_text)


In [ ]:
from tunix_craftext.rollouts.batched import (
    HostBatchPolicy,
    collect_batched_text_rollout_profiled,
    cpu_environment_device_policy,
)

GROUP_SIZE = 4
GROUP_COUNT = 2
HORIZON = 4
BATCH_SIZE = GROUP_SIZE * GROUP_COUNT

profiled_rollout = collect_batched_text_rollout_profiled(
    runtime.adapter,
    renderer,
    backend,
    actions=runtime.actions,
    batch_size=BATCH_SIZE,
    horizon=HORIZON,
    seed=config.run.seed,
    goal=prepared_goal,
    max_new_tokens=min(12, generation.tunix.max_tokens_to_generate),
    invalid_action='fallback',
    fallback_action_id=runtime.actions.index_of('NOOP'),
    host_batch_policy=HostBatchPolicy(prompt_workers=4),
    device_policy=cpu_environment_device_policy(),
)
rollout = profiled_rollout.rollout
print('phase totals ms:', profiled_rollout.phase_totals_ms())
print('event totals:', profiled_rollout.event_totals())
print('first-step actions:', [action.label for action in rollout.decisions[0].actions])


In [ ]:
from tunix_craftext.artifacts.replay import save_replay
from tunix_craftext.rollouts.batched import replays_from_batched_rollout

replays = replays_from_batched_rollout(
    rollout,
    config_path=str(CONFIG_PATH.relative_to(ROOT)),
    commit='notebook-external-vllm-grpo',
    backend=f'{engine.profile.backend}:{engine.profile.model}',
)
for env_index, replay in enumerate(replays):
    path = REPLAY_DIR / f'env-{env_index}.json'
    save_replay(path, replay)
print('saved replays:', len(replays), 'to', REPLAY_DIR)


In [ ]:
import json

from tunix_craftext.training.external_grpo import (
    external_grpo_batch_from_replays,
    save_external_grpo_batch,
    summarize_external_grpo_batch,
)

external_grpo_batch = external_grpo_batch_from_replays(
    goal=prepared_goal,
    group_prefix=f'instruction-{prepared_instruction_index}',
    replays=replays,
    group_size=GROUP_SIZE,
    require_token_provenance=True,
)
summary = summarize_external_grpo_batch(external_grpo_batch)
GRPO_PATH = RUN_DIR / 'external-grpo-batch.json'
SUMMARY_PATH = RUN_DIR / 'external-grpo-summary.json'
TIMING_PATH = RUN_DIR / 'external-grpo-rollout-timing.json'
save_external_grpo_batch(GRPO_PATH, external_grpo_batch)
SUMMARY_PATH.write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
TIMING_PATH.write_text(
    json.dumps(
        {
            'phase_totals_ms': profiled_rollout.phase_totals_ms(),
            'event_totals': profiled_rollout.event_totals(),
            'group_size': GROUP_SIZE,
            'group_count': GROUP_COUNT,
            'horizon': HORIZON,
        },
        indent=2,
        sort_keys=True,
    ),
    encoding='utf-8',
)
print('summary:', summary)
print('saved:', GRPO_PATH)
print('saved:', SUMMARY_PATH)
print('saved:', TIMING_PATH)


In [ ]:
import jax.numpy as jnp

from tunix_craftext.models.tunix_actor import LlmActorTokenScores
from tunix_craftext.research.llm_grpo import evaluate_external_llm_actor_grpo
from tunix_craftext.training.external_grpo import token_batch_from_external_grpo

token_batch = token_batch_from_external_grpo(external_grpo_batch)
behaviour_actor_scores = LlmActorTokenScores(
    token_logprobs=token_batch.old_logprobs,
    entropy=jnp.zeros_like(token_batch.old_logprobs),
    token_mask=token_batch.token_mask,
)
grpo_evaluation = evaluate_external_llm_actor_grpo(
    token_batch,
    behaviour_actor_scores,
    clip_epsilon=0.2,
    entropy_coefficient=0.0,
)
print('token batch shape:', token_batch.token_ids.shape)
print('grpo baseline loss:', float(grpo_evaluation.loss))
print('grpo baseline metrics:', {key: float(value) for key, value in grpo_evaluation.metrics.items()})


In [ ]:
# Release vLLM resources before starting another heavy notebook/trainer in this kernel.
engine.close()
print('closed vLLM engine')
